# Inspección de catálogos

Notebook utilitario para cargar **cualquier** catálogo generado por el pipeline
(`.hdf5` o `.pkl`) como un `pandas.DataFrame` y revisarlo visualmente — sin
necesidad de recordar la estructura interna de cada archivo.

Funciona con:
- `catalogo_icl.hdf5` (salida de `00_catalogo.ipynb`)
- `mary_haloshift_z0.pkl` / `mary_subhaloshift_z0.pkl` (catálogos de shift)
- Cualquier otro `.hdf5`/`.pkl` con datasets/campos 1D o 2D (arrays Nx2, Nx3, Nx6, etc.)

Los campos 2D (ej. posiciones `[N,3]`, `SubhaloMassType` `[N,6]`) se expanden
automáticamente en columnas `campo_0`, `campo_1`, ... para que la tabla sea
completamente plana, al estilo pandas.

In [1]:
import sys, pickle
import numpy as np
import pandas as pd
import h5py
from pathlib import Path
from IPython.display import display

sys.path.insert(0, './original_shift_code')
import params_icl as P

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')

## Funciones de carga genéricas

In [3]:
def _expand_2d(name, arr):
    """Convierte un array (N,) o (N,k) en un dict de columnas planas."""
    arr = np.asarray(arr)
    if arr.ndim == 1:
        return {name: arr}
    if arr.ndim == 2:
        return {f'{name}_{j}': arr[:, j] for j in range(arr.shape[1])}
    # ndim >= 3: no es tabular, se ignora (se reporta aparte)
    return {}


def hdf5_to_dataframe(path, group=None, verbose=True):
    """
    Lee un archivo HDF5 (o un grupo interno) y devuelve un DataFrame plano.
    Los datasets escalares (sin filas) se reportan como atributos, no como columnas.
    """
    cols = {}
    scalars = {}
    skipped = []
    with h5py.File(path, 'r') as f:
        node = f[group] if group else f
        attrs = dict(node.attrs)
        n_rows = None
        for key in node.keys():
            ds = node[key]
            if not isinstance(ds, h5py.Dataset):
                continue
            arr = ds[()]
            if np.ndim(arr) == 0:
                scalars[key] = arr
                continue
            if np.ndim(arr) > 2:
                skipped.append(key)
                continue
            cols.update(_expand_2d(key, arr))
            n_rows = len(arr) if n_rows is None else n_rows
    df = pd.DataFrame(cols)
    if verbose:
        print(f"Archivo: {path}")
        if attrs:
            print(f"  Atributos: {attrs}")
        if scalars:
            print(f"  Escalares (no tabulares): {scalars}")
        if skipped:
            print(f"  Datasets omitidos (ndim>2): {skipped}")
        print(f"  Forma de la tabla: {df.shape}")
    return df


def pkl_to_dataframe(path, verbose=True):
    """
    Lee un catálogo pickle (dict de arrays, como mary_haloshift_z0.pkl) y
    devuelve un DataFrame plano. La clave 'count' se reporta aparte.
    """
    with open(path, 'rb') as f:
        raw = pickle.load(f)

    if not isinstance(raw, dict):
        # pkl es directamente un array/lista
        return pd.DataFrame({'value': np.asarray(raw)})

    cols = {}
    scalars = {}
    skipped = []
    for key, val in raw.items():
        arr = np.asarray(val)
        if arr.ndim == 0:
            scalars[key] = arr.item()
            continue
        if arr.ndim > 2:
            skipped.append(key)
            continue
        cols.update(_expand_2d(key, arr))

    df = pd.DataFrame(cols)
    if verbose:
        print(f"Archivo: {path}")
        if scalars:
            print(f"  Escalares: {scalars}")
        if skipped:
            print(f"  Campos omitidos (ndim>2): {skipped}")
        print(f"  Forma de la tabla: {df.shape}")
    return df


def load_catalog(path, group=None, verbose=True):
    """Dispatcher: detecta la extensión y llama a la función adecuada."""
    path = str(path)
    if path.endswith(('.hdf5', '.h5')):
        return hdf5_to_dataframe(path, group=group, verbose=verbose)
    if path.endswith(('.pkl', '.pickle')):
        return pkl_to_dataframe(path, verbose=verbose)
    raise ValueError(f"Extensión no soportada: {path}")

## 1 · Catálogo ICL final (`catalogo_icl.hdf5`)

Salida de `00_catalogo.ipynb` — un cúmulo por fila.

In [4]:
df_icl = load_catalog(P.CATALOG_OUT)
display(df_icl.head(10).style.background_gradient(cmap='Blues', axis=0))

Archivo: ./catalogo_icl.hdf5
  Atributos: {'basePath': '/home/tnguser/sims.TNG/TNG100-1/output/', 'n_clusters': 166, 'snap': 99}
  Forma de la tabla: (166, 19)


,GroupCM_kpc_0,GroupCM_kpc_1,GroupCM_kpc_2,GroupPos_kpc_0,GroupPos_kpc_1,GroupPos_kpc_2,M200c_Msun,M_bcg_Msun,M_icl_Msun,R200c_kpc,bcg_Mstar_Msun,bcg_pos_kpc_0,bcg_pos_kpc_1,bcg_pos_kpc_2,bcg_sub_idx,concentration,group_idx,icl_frac,t_last_merger_Gyr
0,1099.438110,38627.992188,26353.976562,1253.456543,38864.769531,27025.292969,376813888995328.000000,2873298518016.000000,832864714752.000000,1522.982422,3705585008640.000000,1253.456543,38864.769531,27025.292969,0,5.591064,0,0.224724,6.670852
1,29265.582031,70674.742188,72194.304688,29163.488281,70202.195312,72218.242188,381147007680512.000000,1334234447872.000000,951632068608.000000,1528.820312,2285440073728.000000,29163.488281,70202.195312,72218.242188,17185,3.197941,1,0.416311,0.340075
2,16205.085938,75028.453125,70269.617188,16091.279297,75078.109375,70254.554688,337538124349440.000000,1992806891520.000000,841098133504.000000,1468.077393,2833792892928.000000,16091.279297,75078.109375,70254.554688,31342,5.568976,2,0.296798,10.673787
3,100454.750000,87394.875000,79930.554688,101185.632812,87124.710938,79704.765625,171004676538368.000000,1106580865024.000000,318346657792.000000,1170.387085,1424632053760.000000,101185.632812,87124.710938,79704.765625,41582,8.809569,3,0.223413,10.964058
4,110362.007812,27097.146484,35481.394531,110214.539062,27085.181641,35453.777344,253762757197824.000000,757998026752.000000,444533932032.000000,1334.956665,1202325553152.000000,110214.539062,27085.181641,35453.777344,52618,4.486698,4,0.369665,12.336683
5,27673.312500,100667.812500,3457.293213,27801.535156,100750.929688,3542.686279,203177454469120.000000,896454492160.000000,444793487360.000000,1239.573608,1340881108992.000000,27801.535156,100750.929688,3542.686279,60731,6.850520,5,0.331627,7.446893
6,64355.328125,56204.875000,103777.250000,64127.695312,56373.468750,103623.507812,208590069563392.000000,1766026903552.000000,459275436032.000000,1250.544800,2225182343168.000000,64127.695312,56373.468750,103623.507812,69507,7.978301,6,0.206388,10.673787
7,53932.222656,1279.054443,104151.820312,54284.015625,1384.017334,104079.484375,89359780937728.000000,718241660928.000000,234037035008.000000,942.675842,952085839872.000000,54284.015625,1384.017334,104079.484375,76086,8.433802,7,0.245765,10.518235
8,35084.867188,48787.042969,61017.382812,35234.425781,48828.019531,61132.558594,207975671136256.000000,1580417941504.000000,513936850944.000000,1249.247681,2094143897600.000000,35234.425781,48828.019531,61132.558594,83280,7.192815,8,0.245391,8.514068
9,60027.539062,98609.875000,63534.125000,60109.265625,98744.992188,63557.207031,212519511654400.000000,1656732909568.000000,409934987264.000000,1258.355957,2066222546944.000000,60109.265625,98744.992188,63557.207031,88663,6.528901,9,0.198356,2.787394


**Y aqui estoy probando cargar las propiedades de algun halo en especifico:**

In [18]:
id_halo = 0

In [19]:
icl_halo_test = df_icl[df_icl['group_idx'] == id_halo]

In [20]:
icl_halo_test

,GroupCM_kpc_0,GroupCM_kpc_1,GroupCM_kpc_2,GroupPos_kpc_0,GroupPos_kpc_1,GroupPos_kpc_2,M200c_Msun,M_bcg_Msun,M_icl_Msun,R200c_kpc,bcg_Mstar_Msun,bcg_pos_kpc_0,bcg_pos_kpc_1,bcg_pos_kpc_2,bcg_sub_idx,concentration,group_idx,icl_frac,t_last_merger_Gyr
0,"1,099",3.863e+04,2.635e+04,"1,253",3.886e+04,2.703e+04,3.768e+14,2.873e+12,8.329e+11,"1,523",3.706e+12,"1,253",3.886e+04,2.703e+04,0,5.591,0,0.2247,6.671


## 2 · Catálogo de halos (`mary_haloshift_z0.pkl`)

Un halo por fila (incluye todos los halos candidatos, fossil + no-fossil + aislados).

In [32]:
df_halos = load_catalog(P.CATALOG_PKL)
display(df_halos.head(10))

Archivo: ./mary_haloshift_z0.pkl
  Escalares: {'count': 182}
  Forma de la tabla: (182, 39)


,GroupFirstSub,GroupNsubs,Group_M_Crit200,GroupCM_0,GroupCM_1,GroupCM_2,Group_R_Crit200,GroupPos_0,GroupPos_1,GroupPos_2,GroupGasMetalFractions_0,GroupGasMetalFractions_1,GroupGasMetalFractions_2,GroupGasMetalFractions_3,GroupGasMetalFractions_4,GroupGasMetalFractions_5,GroupGasMetalFractions_6,GroupGasMetalFractions_7,GroupGasMetalFractions_8,GroupGasMetalFractions_9,GroupGasMetallicity,GroupMass,GroupMassType_0,GroupMassType_1,GroupMassType_2,GroupMassType_3,GroupMassType_4,GroupMassType_5,GroupSFR,GroupStarMetallicity,GroupWindMass,GroupNumber,GroupSZPos_0,GroupSZPos_1,GroupSZPos_2,GroupOffset,Group_DM_CM_0,Group_DM_CM_1,Group_DM_CM_2
0,990000000000,17185,2.553e+04,744.8,2.617e+04,1.785e+04,"1,032",849.1,2.633e+04,1.831e+04,0.7524,0.244,0.0003869,0.0001246,0.001845,0.0006052,0.0001765,0.0001652,0.000224,0.0001154,0.003643,3.888e+04,"4,501",3.363e+04,0,0,748,2.54,98.87,0.02123,0.1106,990000000000,753.2,2.618e+04,1.789e+04,0.4359,743,2.616e+04,1.785e+04
1,990000017185,14157,2.582e+04,1.982e+04,4.788e+04,4.89e+04,"1,036",1.976e+04,4.755e+04,4.892e+04,0.7529,0.2437,0.0003618,0.0001158,0.001735,0.0005667,0.0001657,0.0001549,0.0002077,0.0001076,0.003415,3.318e+04,"3,721",2.891e+04,0,0,550.7,1.895,29.32,0.02002,0.04974,990000000001,1.983e+04,4.786e+04,4.891e+04,0.3015,1.982e+04,4.788e+04,4.89e+04
2,990000031342,10240,2.286e+04,1.098e+04,5.082e+04,4.76e+04,994.5,1.09e+04,5.086e+04,4.759e+04,0.7524,0.2439,0.0003843,0.0001234,0.001832,0.0005993,0.0001751,0.000164,0.0002225,0.0001146,0.003615,2.672e+04,"3,338",2.29e+04,0,0,482.8,1.829,14.19,0.02048,0.01851,990000000002,1.097e+04,5.081e+04,4.76e+04,0.0818,1.098e+04,5.083e+04,4.76e+04
3,990000041582,11036,1.158e+04,6.805e+04,5.92e+04,5.414e+04,792.8,6.854e+04,5.902e+04,5.399e+04,0.7529,0.2437,0.0003632,0.0001156,0.001751,0.0005723,0.0001673,0.0001554,0.0002044,0.0001067,0.003436,2.592e+04,"3,170",2.228e+04,0,0,465.7,1.668,277.5,0.0195,0.07196,990000000003,6.812e+04,5.924e+04,5.413e+04,0.6241,6.804e+04,5.92e+04,5.415e+04
4,990000052618,8113,1.719e+04,7.476e+04,1.836e+04,2.404e+04,904.3,7.466e+04,1.835e+04,2.402e+04,0.7534,0.2434,0.000336,0.0001069,0.001625,0.0005289,0.000155,0.0001443,0.0001895,9.917e-05,0.003185,1.975e+04,"2,454",1.699e+04,0,0,308,1.075,42.43,0.01927,0.02944,990000000004,7.474e+04,1.832e+04,2.401e+04,0.09078,7.476e+04,1.836e+04,2.404e+04
5,990000060731,8776,1.376e+04,1.875e+04,6.819e+04,"2,342",839.7,1.883e+04,6.825e+04,"2,400",0.7532,0.2436,0.0003438,0.0001106,0.001656,0.0005397,0.000158,0.0001484,0.0002016,0.000104,0.003262,1.941e+04,"2,142",1.693e+04,0,0,342,1.374,28.6,0.02026,0.05235,990000000005,1.877e+04,6.821e+04,"2,347",0.11,1.874e+04,6.819e+04,"2,341"
6,990000069507,6579,1.413e+04,4.359e+04,3.807e+04,7.03e+04,847.1,4.344e+04,3.819e+04,7.019e+04,0.7525,0.2439,0.0003821,0.0001229,0.001818,0.0005949,0.0001739,0.0001633,0.0002246,0.000115,0.003594,1.817e+04,"2,129",1.571e+04,0,0,334.3,1.361,16.81,0.02049,0.01787,990000000006,4.357e+04,3.81e+04,7.028e+04,0.2067,4.36e+04,3.807e+04,7.03e+04
7,990000076086,7194,"6,053",3.653e+04,866.4,7.055e+04,638.6,3.677e+04,937.5,7.05e+04,0.7524,0.2439,0.0003862,0.0001234,0.001876,0.0006101,0.0001788,0.0001671,0.0002205,0.0001153,0.003678,1.743e+04,"1,987",1.511e+04,0,0,335.4,1.156,58.57,0.01906,0.05102,990000000007,3.667e+04,878.4,7.056e+04,0.2128,3.652e+04,865.1,7.055e+04
8,990000083280,5383,1.409e+04,2.377e+04,3.305e+04,4.133e+04,846.2,2.387e+04,3.308e+04,4.141e+04,0.7523,0.244,0.000393,0.000126,0.001882,0.0006148,0.0001799,0.0001686,0.0002294,0.0001181,0.003712,1.704e+04,"2,074",1.464e+04,0,0,321.4,1.134,24.9,0.01991,0.03429,990000000008,2.378e+04,3.305e+04,4.133e+04,0.148,2.377e+04,3.305e+04,4.133e+04
9,990000088663,7837,1.44e+04,4.066e+04,6.68e+04,4.304e+04,852.4,4.072e+04,6.689e+04,4.305e+04,0.7529,0.2437,0.0003594,0.0001151,0.001742,0.0005672,0.0001661,0.0001551,0.0002056,0.0001072,0.003418,1.728e+04,"2,009",1.498e+04,0,0,293.6,1.103,47.02,0.01939,0.06583,990000000009,4.067e+04,6.682e+04,4.303e+04,0.1052,4.066e+04,6.68e+04,4

In [33]:
# modificacion para homegenizar el catalogo a la snapshot 99
df_halos['GroupNumber'] = df_halos['GroupNumber'] - 990000000000

In [34]:
D_halo_test = df_halos[df_halos['GroupNumber'] == id_halo]

In [35]:
D_halo_test

,GroupFirstSub,GroupNsubs,Group_M_Crit200,GroupCM_0,GroupCM_1,GroupCM_2,Group_R_Crit200,GroupPos_0,GroupPos_1,GroupPos_2,GroupGasMetalFractions_0,GroupGasMetalFractions_1,GroupGasMetalFractions_2,GroupGasMetalFractions_3,GroupGasMetalFractions_4,GroupGasMetalFractions_5,GroupGasMetalFractions_6,GroupGasMetalFractions_7,GroupGasMetalFractions_8,GroupGasMetalFractions_9,GroupGasMetallicity,GroupMass,GroupMassType_0,GroupMassType_1,GroupMassType_2,GroupMassType_3,GroupMassType_4,GroupMassType_5,GroupSFR,GroupStarMetallicity,GroupWindMass,GroupNumber,GroupSZPos_0,GroupSZPos_1,GroupSZPos_2,GroupOffset,Group_DM_CM_0,Group_DM_CM_1,Group_DM_CM_2
0,990000000000,17185,2.553e+04,744.8,2.617e+04,1.785e+04,"1,032",849.1,2.633e+04,1.831e+04,0.7524,0.244,0.0003869,0.0001246,0.001845,0.0006052,0.0001765,0.0001652,0.000224,0.0001154,0.003643,3.888e+04,"4,501",3.363e+04,0,0,748,2.54,98.87,0.02123,0.1106,0,753.2,2.618e+04,1.789e+04,0.4359,743,2.616e+04,1.785e+04


## 3 · Catálogo de subhalos (`mary_subhaloshift_z0.pkl`)

Un subhalo por fila (5000+ filas típicamente — solo se muestran las primeras).

In [36]:
df_subhalos = load_catalog(P.CATALOG_SUBHALOSHIFT_PKL)
display(df_subhalos.head(10))

Archivo: ./mary_subhaloshift_z0.pkl
  Escalares: {'count': 5225}
  Forma de la tabla: (5225, 25)


,SubhaloFlag,SubhaloCM_0,SubhaloCM_1,SubhaloCM_2,SubhaloPos_0,SubhaloPos_1,SubhaloPos_2,SubhaloGrNr,SubhaloMassType_0,SubhaloMassType_1,SubhaloMassType_2,SubhaloMassType_3,SubhaloMassType_4,SubhaloMassType_5,SubhaloSFR,SubhaloMass,SubhaloStellarPhotometrics_0,SubhaloStellarPhotometrics_1,SubhaloStellarPhotometrics_2,SubhaloStellarPhotometrics_3,SubhaloStellarPhotometrics_4,SubhaloStellarPhotometrics_5,SubhaloStellarPhotometrics_6,SubhaloStellarPhotometrics_7,SubhaloNumber
0,True,832.4,2.637e+04,1.806e+04,849.1,2.633e+04,1.831e+04,990000000000,"3,901",2.333e+04,0,0,251,0.7018,2.227,2.748e+04,-24.1,-24.51,-25.38,-28.31,-24.97,-25.72,-26.07,-26.33,990000000000
1,True,124.5,2.461e+04,1.686e+04,106.5,2.463e+04,1.69e+04,990000000000,444.6,"3,184",0,0,37.85,0.1256,0.4133,"3,667",-22.11,-22.5,-23.35,-26.25,-22.95,-23.68,-24.03,-24.28,990000000001
2,True,850.4,2.671e+04,1.751e+04,853.1,2.673e+04,1.751e+04,990000000000,12.61,718.2,0,0,42.48,0.1775,17.39,773.5,-23.07,-23.13,-23.78,-26.55,-23.5,-24.07,-24.37,-24.6,990000000002
3,True,243,2.65e+04,1.589e+04,245.5,2.652e+04,1.589e+04,990000000000,16.97,310.5,0,0,11.62,0.03442,5.079,339.1,-21.97,-22.05,-22.68,-25.39,-22.41,-22.96,-23.24,-23.45,990000000003
4,True,766.5,2.652e+04,1.553e+04,768.8,2.652e+04,1.553e+04,990000000000,9.439,306.8,0,0,9.032,0.05234,1.415,325.4,-20.95,-21.15,-21.88,-24.67,-21.55,-22.19,-22.51,-22.74,990000000004
5,True,"1,426",2.634e+04,1.904e+04,"1,427",2.634e+04,1.904e+04,990000000000,2.06,282.9,0,0,13.02,0.05648,0.8866,298,-21.08,-21.42,-22.23,-25.13,-21.85,-22.56,-22.9,-23.15,990000000005
6,True,904.3,2.661e+04,1.783e+04,904.8,2.661e+04,1.783e+04,990000000000,1.744,185.4,0,0,19.6,0.07283,2.184,206.8,-21.55,-21.82,-22.64,-25.57,-22.26,-22.96,-23.31,-23.57,990000000006
7,True,"1,410",2.666e+04,1.728e+04,"1,411",2.666e+04,1.728e+04,990000000000,0.01707,124.7,0,0,13.05,0.04237,0,137.8,-20.67,-21.15,-22.05,-25.07,-21.62,-22.4,-22.78,-23.05,990000000007
8,True,835.7,2.651e+04,1.745e+04,835.9,2.651e+04,1.745e+04,990000000000,0.0129,103.3,0,0,15.12,0.07196,0,118.5,-20.74,-21.23,-22.14,-25.2,-21.7,-22.5,-22.88,-23.16,990000000008
9,True,473.1,2.657e+04,1.805e+04,471.9,2.657e+04,1.805e+04,990000000000,0,115.4,0,0,13.62,0.03107,0,129.1,-20.83,-21.3,-22.19,-25.18,-21.76,-22.54,-22.9,-23.17,990000000009


In [37]:
# Masa estelar en log10(M_sun), columna derivada para inspección rápida
if 'SubhaloMassType_4' in df_subhalos:
    df_subhalos['logMstar'] = np.log10(df_subhalos['SubhaloMassType_4'] / P.h + 1e-30) + 10
    display(df_subhalos[['SubhaloGrNr', 'logMstar']].sort_values('logMstar', ascending=False).head(10))

,SubhaloGrNr,logMstar
0,990000000000,12.57
476,990000000002,12.45
270,990000000001,12.36
1117,990000000006,12.35
1387,990000000008,12.32
1484,990000000009,12.32
1619,990000000010,12.24
1718,990000000011,12.2
2438,990000000022,12.19
654,990000000003,12.15


## 4 · Inspección rápida (histogramas)

Usa directamente `DataFrame.plot` / `DataFrame.hist` de pandas — sin necesidad
de escribir matplotlib manualmente para una mirada rápida.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 100})

if 'M200c_Msun' in df_icl:
    np.log10(df_icl['M200c_Msun']).plot.hist(bins=20, color='steelblue', edgecolor='white',
                                              title='log10(M200c) — catalogo_icl.hdf5')
    plt.xlabel(r'$\log_{10}(M_{200c}/M_\odot)$')
    plt.show()

## 5 · Cargar cualquier otro archivo

Cambia `mi_path` por la ruta del catálogo que quieras inspeccionar
(funciona para cualquier `.hdf5`/`.pkl` generado por este pipeline,
incluyendo `fossil_cat_TNG50.hdf5` u otros).

In [ ]:
mi_path = P.CATALOG_OUT   # ← cambia esta ruta
df_generico = load_catalog(mi_path)
display(df_generico.head())